<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_09-2_RPG_MCP_Server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 2 | AI RPG 게임 마스터: MCP로 게임 서버 만들기

**실행 환경:** Google Colab · **API 키 불필요**

> **핵심 질문:** 실습 1에서 만든 `attack_enemy()` 같은 함수를 다른 AI 앱도 공통 방식으로 발견하고 호출하려면 어떻게 해야 할까요?

이번 실습에서는 게임 함수를 **MCP Server의 Tool**로 노출하고, **Resource**(몬스터 설명)와 **Prompt**(전투 전략 템플릿)를 추가합니다. 실제 LLM은 사용하지 않습니다. **MCP 표준 통신과 도구 실행을 먼저 검증하는 수업**입니다.

### 학습 목표
1. MCP의 **Server / Client / Host** 역할을 설명할 수 있다.
2. `@mcp.tool()`, `@mcp.resource()`, `@mcp.prompt()`를 구별한다.
3. `Client(mcp)`로 서버의 도구를 발견하고 정상/오류 호출을 검증한다.
4. 읽기 작업과 상태 변경 작업의 차이를 테스트한다.

**전체 주석판:** Python 실행문, MCP 도구 등록, 요청/응답 처리, 검증문에 쉬운 한글 주석을 추가했습니다. 기존 코드는 변경하지 않았습니다.


## 0. Colab 실행 안내

- 위에서 아래로 순서대로 실행하세요. `런타임 → 모두 실행`을 권장합니다.
- 설치 셀은 인터넷 연결이 필요합니다. 설치 이후 기본 실습은 API 키나 외부 게임 서버가 필요하지 않습니다.
- **In-memory Client**를 사용하므로 서버 포트 열기·별도 프로세스 실행은 필요하지 않습니다.
- 실습 1과 **같은 기본 게임 규칙**을 사용하지만, 이 노트북만으로 독립 실행할 수 있도록 게임 로직을 다시 정의합니다.
- 기준 패키지: 강의자료의 `mcp==2.3.0`. 배포된 SDK 버전에 따라 API가 바뀔 수 있으므로 버전을 고정합니다.

**검증 범위 안내:** 노트북 생성 환경에서는 패키지 설치용 네트워크가 차단되어 MCP 호출을 실제로 실행하지 못했습니다. 아래 자동 테스트는 Colab에서 직접 확인하세요.

In [ ]:
# Colab에는 mcp 라이브러리가 기본 설치되어 있지 않을 수 있습니다.
# 강의자료와 일치하도록 버전을 고정해 환경 차이를 줄입니다.
# 수업에서 사용할 MCP Python SDK 2.3.0을 설치합니다. -q는 설치 로그를 간단히 표시합니다.
%pip -q install "mcp==2.3.0"


In [ ]:
# 설치된 버전을 확인합니다. 예상: 2.3.0
# 설치된 패키지 버전을 읽는 함수를 가져옵니다.
from importlib.metadata import version
# MCP 서버 객체를 만들 클래스를 가져옵니다.
from mcp.server import MCPServer
# MCP 서버에 접속할 테스트 클라이언트를 가져옵니다.
from mcp import Client
# Resource 응답이 텍스트인지 검사할 타입을 가져옵니다.
from mcp.types import TextResourceContents

# 설치된 MCP SDK 버전을 화면에 표시합니다.
print("mcp SDK 버전:", version("mcp"))
# 예상 버전과 다르면 이후 예제의 API가 다를 수 있으므로 중단합니다.
assert version("mcp") == "2.3.0", "강의 실습 버전을 확인하세요."


mcp SDK 버전: 2.3.0


## 1. 게임 상태 만들기

슬라임과 싸우는 간단한 게임을 구현합니다. **게임 상태는 Python 코드가 보관**합니다. MCP Tool은 이 상태를 조회하거나 변경하는 서버 함수를 외부에 제공합니다.

| 항목 | 시작 값 | 규칙 |
|---|---:|---|
| 플레이어 HP | 100 | 0이면 패배 |
| 슬라임 HP | 60 | 0이면 승리 |
| 포션 | 2개 | 사용 시 HP 30 회복, 최대 100 |
| 공격 | 20 피해 | 공격 1회마다 턴 1 증가 |
| 최대 턴 | 10 | 미종료 시 강제 종료 |

> 단순화를 위해 **몬스터 반격은 구현하지 않습니다**. 실제 대전 규칙 대신 MCP의 호출 흐름에 집중하기 위한 교육용 규칙입니다.

In [ ]:
# 게임 상태를 쉽게 정의하고 딕셔너리로 바꾸기 위한 도구를 가져옵니다.
from dataclasses import dataclass, asdict
# 함수의 반환값이 여러 형태일 수 있음을 나타내는 타입을 가져옵니다.
from typing import Any
# JSON 데이터를 다루기 위한 표준 라이브러리를 가져옵니다.
import json

# GameState를 자동 생성자와 함께 제공하는 데이터 클래스로 선언합니다.
@dataclass
# 플레이어와 몬스터의 현재 상태를 하나로 묶는 자료형입니다.
class GameState:
    # 서버가 보유하는 게임의 원본 상태입니다.
    # 플레이어의 시작 체력을 100으로 지정합니다.
    player_hp: int = 100
    # 몬스터 슬라임의 시작 체력을 60으로 지정합니다.
    enemy_hp: int = 60
    # 시작할 때 회복 포션을 2개 지급합니다.
    potions: int = 2
    # 아직 행동하지 않았으므로 턴 번호는 0입니다.
    turn: int = 0
    # 게임이 진행 중인지, 승리·패배·시간 초과인지 저장합니다.
    outcome: str = "playing"  # playing / win / lose / timeout

# 하나의 게임 인스턴스를 메모리에 생성합니다.
# 새 GameState 객체를 생성해 게임을 처음 상태로 만듭니다.
game = GameState()

# 게임을 처음부터 다시 시작하는 내부 함수를 정의합니다.
def reset_game_logic() -> dict[str, Any]:
    """게임을 새로 시작합니다. 테스트 사이의 상태 간섭을 방지할 때 사용합니다."""
    # 함수 안에서 전역 game 변수를 새 객체로 바꿀 것임을 알립니다.
    global game
    # 기존 게임 상태를 새 초기 상태로 교체합니다.
    game = GameState()
    # 성공 여부와 초기 상태를 딕셔너리로 반환합니다.
    return {"ok": True, "state": asdict(game)}

# 현재 게임 상태를 읽기만 하는 함수를 정의합니다.
def status_logic() -> dict[str, Any]:
    """현재 상태만 조회합니다. 이 함수는 턴이나 HP를 바꾸지 않습니다."""
    # game 값을 수정하지 않고 현재 상태만 반환합니다.
    return {"ok": True, "state": asdict(game)}

# 사용자가 요청한 공격 대상 이름을 받아 전투를 처리합니다.
def attack_logic(target: str) -> dict[str, Any]:
    """서버가 인자와 게임 규칙을 검증한 뒤에만 공격을 실행합니다."""
    # 허용된 대상인 slime이 아니라면 공격을 거부합니다.
    if target != "slime":
        # 잘못된 대상이라는 오류와 사용 가능한 대상 목록을 반환합니다.
        return {"ok": False, "error_code": "INVALID_TARGET", "allowed_targets": ["slime"]}
    # 이미 승리하거나 종료한 게임인지 확인합니다.
    if game.outcome != "playing":
        # 끝난 게임에서는 상태를 바꾸지 않고 오류를 반환합니다.
        return {"ok": False, "error_code": "GAME_OVER", "outcome": game.outcome}

    # 실제 상태 변경: 모델의 요청이 아니라 서버의 Python 코드가 수행합니다.
    # 몬스터 HP를 20 줄이되 0보다 작아지지 않도록 제한합니다.
    game.enemy_hp = max(0, game.enemy_hp - 20)
    # 공격에 성공했으므로 사용한 턴 수를 1 증가시킵니다.
    game.turn += 1
    # 몬스터 HP가 0이 되면 승리 조건을 충족합니다.
    if game.enemy_hp == 0:
        # 게임 종료 상태를 승리(win)로 기록합니다.
        game.outcome = "win"
    # 아직 이기지 못한 상태에서 최대 10턴을 사용했는지 검사합니다.
    elif game.turn >= 10:
        # 허용된 턴을 모두 사용했으므로 시간 초과로 끝냅니다.
        game.outcome = "timeout"
    # 공격 결과와 갱신된 전체 상태를 반환합니다.
    return {"ok": True, "action": "attack", "state": asdict(game)}

# 포션으로 플레이어 체력을 회복하는 함수를 정의합니다.
def heal_logic() -> dict[str, Any]:
    """포션이 있고 HP가 부족할 때만 회복합니다."""
    # 게임이 끝났다면 회복을 허용하지 않습니다.
    if game.outcome != "playing":
        # 게임 종료 오류를 반환해 다음 호출에서 원인을 알 수 있게 합니다.
        return {"ok": False, "error_code": "GAME_OVER"}
    # 남은 포션 수가 0 이하인지 확인합니다.
    if game.potions <= 0:
        # 포션 부족 오류를 구체적인 코드로 반환합니다.
        return {"ok": False, "error_code": "NO_POTION", "potions": 0}
    # 체력이 이미 최대치이면 포션을 낭비하지 않도록 검사합니다.
    if game.player_hp >= 100:
        # 회복할 필요가 없다는 오류 코드를 반환합니다.
        return {"ok": False, "error_code": "HP_ALREADY_FULL"}
    # 체력을 30 회복하되 최대 HP 100을 넘지 않도록 합니다.
    game.player_hp = min(100, game.player_hp + 30)
    # 사용한 포션 1개를 차감합니다.
    game.potions -= 1
    # 회복도 한 턴의 행동이므로 턴 수를 증가시킵니다.
    game.turn += 1
    # 회복 후 최대 턴에 도달했는지 확인합니다.
    if game.turn >= 10:
        # 최대 턴을 소진했다면 시간 초과 상태로 저장합니다.
        game.outcome = "timeout"
    # 회복 성공과 변경된 체력·포션·턴 정보를 반환합니다.
    return {"ok": True, "action": "heal", "state": asdict(game)}

# 게임이 초기화되었는지 상태를 출력해 확인합니다.
print("초기 게임 상태:", status_logic()["state"])


초기 게임 상태: {'player_hp': 100, 'enemy_hp': 60, 'potions': 2, 'turn': 0, 'outcome': 'playing'}


## 2. MCP Server 등록: Tool

- **Tool:** 모델 또는 Host가 필요할 때 호출할 수 있는 함수입니다.
- `name`은 Python 함수명에서, `description`은 docstring에서, 입력 스키마는 타입 힌트에서 만들어집니다.
- MCP 서버가 도구를 제공한다고 해서 **모델이 직접 Python 함수를 실행하는 것은 아닙니다**. MCP Client가 프로토콜 메시지를 보내고 서버 코드가 실행합니다.

읽기 전용인 `get_game_status`와 상태를 변경하는 `attack_enemy`·`heal_player`를 분리합니다.

In [ ]:
# RPG 기능을 공개할 MCP 서버 객체를 만듭니다.
# 게임 함수를 외부에 공개할 MCP 서버 객체를 생성합니다.
mcp = MCPServer("RPG Game Server")

# 이어서 정의하는 함수를 MCP Tool로 공개합니다.
@mcp.tool()
# 현재 상태를 읽는 MCP 도구의 이름과 반환 타입을 정의합니다.
def get_game_status() -> dict[str, Any]:
    """현재 플레이어·몬스터 HP, 포션 수와 턴을 조회한다. 게임 상태는 바꾸지 않는다."""
    # 내부 조회 함수를 호출하고 결과를 클라이언트에 전달합니다.
    return status_logic()

# 공격 함수를 MCP Tool로 등록합니다.
@mcp.tool()
# 공격 대상 target을 인자로 받는 도구입니다.
def attack_enemy(target: str) -> dict[str, Any]:
    """지정한 몬스터를 공격한다. 현재 지원되는 target은 'slime'이다."""
    # 내부 공격 로직에 입력 인자를 전달해 실행합니다.
    return attack_logic(target)

# 회복 함수를 MCP Tool로 등록합니다.
@mcp.tool()
# 인자가 없는 회복 도구를 정의합니다.
def heal_player() -> dict[str, Any]:
    """포션 한 개를 사용해 플레이어 HP를 최대 30 회복한다."""
    # 내부 회복 로직을 실행하고 결과를 반환합니다.
    return heal_logic()

# 게임 초기화 함수를 MCP Tool로 등록합니다.
@mcp.tool()
# 서버의 진행 상태를 다시 시작하는 쓰기 도구를 정의합니다.
def reset_game() -> dict[str, Any]:
    """게임을 초기 상태로 재설정한다. 기존 진행 상태를 덮어쓰는 쓰기 도구다."""
    # 실제 상태 초기화는 내부 Python 함수가 담당합니다.
    return reset_game_logic()

# 등록한 Tool이 4개임을 알려 주는 안내문입니다.
print("MCP Server 등록 완료: Tool 4개")


MCP Server 등록 완료: Tool 4개


## 3. MCP Server 등록: Resource와 Prompt

**Resource**는 URI로 읽는 데이터입니다. 이 예제의 몬스터 설명은 게임 상태와 관계없는 **고정된 설정 문서**입니다.  
**Prompt**는 Host/사용자가 불러다 사용할 수 있는 **재사용 가능한 지시문 템플릿**입니다. 템플릿을 가져온 것만으로 LLM이 실행되지는 않습니다.

- Resource: `game://monsters/slime`
- Prompt: `battle_strategy(style)`

In [ ]:
# 고유 URI를 가진 읽기 전용 Resource를 MCP 서버에 등록합니다.
@mcp.resource("game://monsters/slime")
# 몬스터 도감 텍스트를 제공하는 함수를 정의합니다.
def slime_info() -> str:
    """슬라임의 기본 설정을 알려준다. 현재 전투 HP가 아닌 몬스터 도감 정보다."""
    # 현재 HP가 아닌 몬스터의 기본 규칙을 문자열로 반환합니다.
    return "슬라임: 초기 HP 60, 플레이어 공격 1회당 피해 20. 세 번 맞으면 쓰러진다."

# 재사용 가능한 프롬프트 템플릿을 MCP 서버에 등록합니다.
@mcp.prompt()
# style 인자로 전투 조언 스타일을 바꾸는 프롬프트입니다.
def battle_strategy(style: str = "safe") -> str:
    """원하는 전투 스타일에 맞춰 전략을 요청하는 프롬프트 템플릿."""
    # 여러 줄의 문구를 하나의 문자열로 합쳐 반환하기 시작합니다.
    return (
        # 입력받은 스타일을 프롬프트 안내문에 삽입합니다.
        f"당신은 RPG 전투 전략 조언자다. 스타일: {style}. "
        # 전투 정보와 포션·턴 제한을 확인하도록 요청합니다.
        "전투 상태를 먼저 확인하고, 포션과 턴 제한을 고려해 다음 행동을 짧게 제안하라. "
        # 도구 결과를 보지 않고 행동 완료를 주장하지 않도록 안내합니다.
        "도구 결과를 확인하지 않은 행동을 완료했다고 주장하지 마라."
    # 여러 줄로 표현한 문자열 반환 구문을 닫습니다.
    )

# Resource와 Prompt 등록 완료 메시지를 출력합니다.
print("등록 완료: Resource 1개 + Prompt 1개")


등록 완료: Resource 1개 + Prompt 1개


## 4. In-memory MCP Client 연결 및 도구 목록 확인

`Client(mcp)`는 서버 객체와 같은 Python 프로세스에서 통신합니다. **HTTP 주소·포트·서버 파일이 필요 없습니다.**

실습 1에서 작성한 수동 함수 목록과 달리, 이번에는 **MCP의 도구 발견 기능**으로 서버가 제공하는 이름과 스키마를 얻습니다.

In [ ]:
# Colab/Jupyter는 최상위 await를 지원합니다.
# mcp 객체를 직접 넘겨 테스트용 In-memory 연결을 만듭니다.
# 같은 프로세스의 서버 객체에 연결하고 사용 후 연결을 자동 종료합니다.
async with Client(mcp) as client:
    # 서버에서 공개한 Tool 목록을 조회합니다.
    # MCP tools/list 요청을 보내 공개된 도구 목록을 가져옵니다.
    tool_listing = await client.list_tools()
    # 서버에서 제공한 각 Tool을 차례대로 살펴봅니다.
    for tool in tool_listing.tools:
        # 도구의 이름을 출력합니다.
        print(f"이름: {tool.name}")
        # 도구의 기능 설명을 출력합니다.
        print(f"설명: {tool.description}")
        # 도구 인자의 JSON Schema를 한글이 깨지지 않게 출력합니다.
        print("입력 스키마:", json.dumps(tool.input_schema, ensure_ascii=False))
        # 도구 간 구분을 위해 구분선을 출력합니다.
        print("-" * 55)

    # 받은 Tool 목록에서 이름만 집합으로 모읍니다.
    tool_names = {tool.name for tool in tool_listing.tools}
    # 필수 도구 4개가 모두 등록되었는지 확인합니다.
    assert {"attack_enemy", "heal_player", "get_game_status", "reset_game"} <= tool_names

# 목록과 스키마 검사가 성공했음을 출력합니다.
print("검증 성공: MCP Tool 4개 검색")


이름: get_game_status
설명: 현재 플레이어·몬스터 HP, 포션 수와 턴을 조회한다. 게임 상태는 바꾸지 않는다.
입력 스키마: {"type": "object", "properties": {}, "title": "get_game_statusArguments"}
-------------------------------------------------------
이름: attack_enemy
설명: 지정한 몬스터를 공격한다. 현재 지원되는 target은 'slime'이다.
입력 스키마: {"type": "object", "properties": {"target": {"title": "Target", "type": "string"}}, "required": ["target"], "title": "attack_enemyArguments"}
-------------------------------------------------------
이름: heal_player
설명: 포션 한 개를 사용해 플레이어 HP를 최대 30 회복한다.
입력 스키마: {"type": "object", "properties": {}, "title": "heal_playerArguments"}
-------------------------------------------------------
이름: reset_game
설명: 게임을 초기 상태로 재설정한다. 기존 진행 상태를 덮어쓰는 쓰기 도구다.
입력 스키마: {"type": "object", "properties": {}, "title": "reset_gameArguments"}
-------------------------------------------------------
검증 성공: MCP Tool 4개 검색


## 5. MCP Client로 전투 실행하기

한 번의 요청에서 데이터가 이동하는 경로는 **Client → tools/call → Server의 Python 함수 → 결과 → Client**입니다.

결과의 두 필드를 구분하세요.
- `is_error`: MCP 도구 실행 자체의 오류 여부
- `structured_content`: 코드에서 처리할 수 있는 구조화 결과

**중요:** 예제의 `ok: False`는 예상 가능한 **게임 업무 오류**이므로, MCP 호출 자체가 실패했다는 뜻은 아닙니다.

In [ ]:
# In-memory Client를 열어 MCP 서버에 연결합니다.
async with Client(mcp) as client:
    # 결과가 이전 실행에 의존하지 않도록 먼저 초기화합니다.
    # 이전 테스트 영향을 제거하기 위해 reset_game을 호출합니다.
    reset_result = await client.call_tool("reset_game", {})
    # MCP 수준의 실행 오류가 없었는지 검증합니다.
    assert not reset_result.is_error

    # ① 상태 조회: 읽기 작업이므로 턴이 증가하지 않아야 합니다.
    # 상태를 읽는 도구를 호출합니다. 조회는 HP와 턴을 바꾸지 않습니다.
    before = await client.call_tool("get_game_status", {})
    # 공격 이전의 구조화된 상태 데이터를 출력합니다.
    print("공격 전:", before.structured_content)
    # 시작 시 몬스터 HP가 60인지 확인합니다.
    assert before.structured_content["state"]["enemy_hp"] == 60

    # ② 공격 도구 호출: 슬라임 HP가 60 -> 40이 됩니다.
    # tools/call로 슬라임 공격 함수를 요청합니다.
    attacked = await client.call_tool("attack_enemy", {"target": "slime"})
    # 공격 결과를 화면에 출력합니다.
    print("공격 후:", attacked.structured_content)
    # 도구 호출 자체가 오류 없이 끝났는지 확인합니다.
    assert attacked.is_error is False
    # 실제 서버 함수 실행으로 몬스터 HP가 40이 되었는지 검사합니다.
    assert attacked.structured_content["state"]["enemy_hp"] == 40

    # ③ 상태 재조회: 서버 메모리의 HP 변경이 유지되는지 검사합니다.
    # 조회 도구를 다시 호출하여 서버의 최신 상태를 읽습니다.
    after = await client.call_tool("get_game_status", {})
    # 변경된 몬스터 HP가 서버 상태에 유지되었는지 확인합니다.
    assert after.structured_content["state"]["enemy_hp"] == 40
    # 공격 행동이 한 턴을 소비했는지도 확인합니다.
    assert after.structured_content["state"]["turn"] == 1

# 상태 변경 및 재조회 검증 완료 메시지를 출력합니다.
print("검증 성공: 공격이 실제 게임 상태를 변경함")


공격 전: {'ok': True, 'state': {'player_hp': 100, 'enemy_hp': 60, 'potions': 2, 'turn': 0, 'outcome': 'playing'}}
공격 후: {'ok': True, 'action': 'attack', 'state': {'player_hp': 100, 'enemy_hp': 40, 'potions': 2, 'turn': 1, 'outcome': 'playing'}}
검증 성공: 공격이 실제 게임 상태를 변경함


## 6. Resource 읽기 · Prompt 가져오기

이 두 기능은 Tool 호출과 구분됩니다.

- `read_resource(uri)`는 **지정된 URI의 내용을 읽습니다**.
- `get_prompt(name, arguments)`는 **문장을 만들어 돌려줍니다**. 전략을 실행하거나 적을 공격하지는 않습니다.

In [ ]:
# 서버와 연결하여 Resource와 Prompt를 조회합니다.
async with Client(mcp) as client:
    # 서버가 제공하는 고정 URI 목록을 확인합니다.
    # 등록된 Resource의 목록을 가져옵니다.
    resources = await client.list_resources()
    # 각 Resource의 URI만 추려 보기 쉽게 출력합니다.
    print("Resource 목록:", [str(item.uri) for item in resources.resources])

    # 하나의 Resource를 읽고 텍스트를 확인합니다.
    # 몬스터 도감 URI의 Resource 내용을 읽습니다.
    resource_result = await client.read_resource("game://monsters/slime")
    # 응답 요소 가운데 텍스트만 목록으로 추출합니다.
    resource_texts = [part.text for part in resource_result.contents
                      # 이미지 등 다른 콘텐츠 타입은 제외하고 텍스트 타입만 선택합니다.
                      if isinstance(part, TextResourceContents)]
    # 읽어 온 몬스터 도감 설명을 출력합니다.
    print("슬라임 도감:", resource_texts)
    # 도감 내용에 슬라임 정보가 포함되는지 확인합니다.
    assert any("슬라임" in text for text in resource_texts)

    # Prompt 목록에서 이름을 찾고 style 인자를 적용합니다.
    # MCP 서버에 등록된 Prompt 목록을 가져옵니다.
    prompts = await client.list_prompts()
    # 등록된 Prompt 이름들을 출력합니다.
    print("Prompt 목록:", [item.name for item in prompts.prompts])
    # style=aggressive를 넣어 전투 전략 템플릿을 가져옵니다.
    prompt_result = await client.get_prompt("battle_strategy", {"style": "aggressive"})
    # 프롬프트 내용을 표시할 제목을 출력합니다.
    print("생성된 프롬프트:")
    # 프롬프트 응답에 포함된 메시지를 하나씩 처리합니다.
    for message in prompt_result.messages:
        # 메시지 콘텐츠에 text 속성이 있는지 확인합니다.
        if hasattr(message.content, "text"):
            # 프롬프트의 발화 역할과 문장을 출력합니다.
            print(message.role, ":", message.content.text)

    # battle_strategy가 프롬프트 목록에 있는지 검사합니다.
    assert "battle_strategy" in [item.name for item in prompts.prompts]

# Resource와 Prompt 검증 성공 메시지를 출력합니다.
print("검증 성공: Resource와 Prompt 조회")


Resource 목록: ['game://monsters/slime']
슬라임 도감: ['슬라임: 초기 HP 60, 플레이어 공격 1회당 피해 20. 세 번 맞으면 쓰러진다.']
Prompt 목록: ['battle_strategy']
생성된 프롬프트:
user : 당신은 RPG 전투 전략 조언자다. 스타일: aggressive. 전투 상태를 먼저 확인하고, 포션과 턴 제한을 고려해 다음 행동을 짧게 제안하라. 도구 결과를 확인하지 않은 행동을 완료했다고 주장하지 마라.
검증 성공: Resource와 Prompt 조회


## 7. 실패 상황 테스트: 업무 오류와 입력 오류

**업무 오류:** 대상 몬스터가 없거나 포션이 부족하면 `ok: False`와 `error_code`를 반환합니다.  
**입력 오류:** `target` 인자를 누락하면 SDK의 입력 검증에서 거부될 수 있습니다. 이때 구체적인 오류 표현은 SDK 설정에 따라 `is_error` 결과 또는 예외가 될 수 있으므로 양쪽을 확인합니다.

> 보안 주의: 서버가 스스로 작성한 설명/Annotation은 강제 권한 정책이 아닙니다. 실제 서비스에서는 Host 및 실행 환경의 승인·권한 정책을 따로 구성해야 합니다.

In [ ]:
# Client를 열어 오류 상황을 재현합니다.
async with Client(mcp) as client:
    # 오류 테스트 전에 게임 상태를 원상복구합니다.
    await client.call_tool("reset_game", {})

    # ① 허용되지 않는 몬스터는 서버의 업무 규칙으로 거부됩니다.
    # 지원하지 않는 dragon을 대상으로 공격을 요청합니다.
    wrong_target = await client.call_tool("attack_enemy", {"target": "dragon"})
    # 반환된 업무 오류 데이터를 화면에 표시합니다.
    print("잘못된 대상:", wrong_target.structured_content)
    # 오류 코드가 INVALID_TARGET인지 확인합니다.
    assert wrong_target.structured_content["error_code"] == "INVALID_TARGET"
    # 업무 처리 성공 여부가 False인지 확인합니다.
    assert wrong_target.structured_content["ok"] is False
    # MCP 통신은 성공했지만 게임 규칙 검증에서 거부된 사례입니다.
    assert wrong_target.is_error is False  # 정상적인 MCP 호출이지만 업무 처리 실패

    # ② 최대 HP에서 회복하면 포션을 소비하지 않고 업무 오류를 반환합니다.
    # HP가 100인 상태에서 회복을 시도합니다.
    full_hp = await client.call_tool("heal_player", {})
    # 최대 체력에서의 회복 결과를 출력합니다.
    print("최대 HP에서 회복:", full_hp.structured_content)
    # 포션을 쓰지 않았다는 의미의 오류 코드를 확인합니다.
    assert full_hp.structured_content["error_code"] == "HP_ALREADY_FULL"

    # ③ 필수 인자 누락은 SDK가 거부해야 합니다. 반환 방식은 환경별로 확인합니다.
    # SDK의 스키마 검증 결과가 오류 응답인지 예외인지 확인합니다.
    try:
        # 필수 target 인자를 생략해 호출해 봅니다.
        missing = await client.call_tool("attack_enemy", {})
        # 오류 응답의 is_error 여부를 화면에 표시합니다.
        print("필수 인자 누락: is_error =", missing.is_error)
        # 오류 상세 내용을 출력합니다.
        print("메시지 =", missing.content)
        # 반환된 응답이 오류로 표시되었는지 검증합니다.
        assert missing.is_error is True
    # 호출이 예외를 던졌다면 이 분기에서 처리합니다.
    except Exception as exc:
        # 예외의 클래스와 일부 메시지를 출력합니다.
        print("필수 인자 누락: SDK 예외 =", type(exc).__name__, str(exc)[:200])

    # 잘못된 호출 후 게임 상태가 바뀌지 않아야 합니다.
    # 실패한 요청 이후의 게임 상태를 다시 읽습니다.
    current = await client.call_tool("get_game_status", {})
    # 잘못된 호출이 몬스터 HP를 바꾸지 않았는지 검사합니다.
    assert current.structured_content["state"]["enemy_hp"] == 60
    # 잘못된 호출이 턴 수를 증가시키지 않았는지 검사합니다.
    assert current.structured_content["state"]["turn"] == 0

# 업무 오류와 인자 오류 테스트가 끝났음을 알립니다.
print("검증 성공: 잘못된 입력에서 게임 상태가 변경되지 않음")


잘못된 대상: {'ok': False, 'error_code': 'INVALID_TARGET', 'allowed_targets': ['slime']}
최대 HP에서 회복: {'ok': False, 'error_code': 'HP_ALREADY_FULL'}
필수 인자 누락: is_error = True
메시지 = [TextContent(type='text', text='Error executing tool attack_enemy: 1 validation error for attack_enemyArguments\ntarget\n  Field required [type=missing, input_value={}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.14/v/missing', annotations=None, meta=None)]
검증 성공: 잘못된 입력에서 게임 상태가 변경되지 않음


## 8. 자동 Contract Test: 정상·경계·반복·오류

다음 셀은 테스트를 한 번에 수행합니다. 테스트 결과가 성공하면 `총 7개 체크 통과`가 출력됩니다.  
서버 상태가 테스트끼리 영향을 주지 않도록 각 시나리오 시작 시 `reset_game()`을 호출합니다.

In [ ]:
# 여러 MCP Tool 호출의 계약을 한 번에 검증하는 비동기 함수입니다.
async def run_contract_tests() -> None:
    # 성공한 검사 수를 기록할 변수를 0으로 시작합니다.
    passed = 0

    # 테스트에서 사용할 In-memory MCP 연결을 엽니다.
    async with Client(mcp) as client:
        # 1) 초기 상태
        # 처음부터 테스트하기 위해 서버 상태를 리셋합니다.
        initial = await client.call_tool("reset_game", {})
        # 초기 몬스터 HP가 60인지 확인합니다.
        assert initial.structured_content["state"]["enemy_hp"] == 60
        # 첫 번째 검사가 통과했으므로 성공 횟수를 올립니다.
        passed += 1

        # 2) 읽기 도구는 상태를 변경하지 않음
        # 변경 전 게임 상태를 조회합니다.
        first = await client.call_tool("get_game_status", {})
        # 상태가 바뀌지 않았는지 비교할 두 번째 조회를 실행합니다.
        second = await client.call_tool("get_game_status", {})
        # 읽기 도구는 상태를 수정하지 않는다는 계약을 확인합니다.
        assert first.structured_content["state"] == second.structured_content["state"]
        # 두 번째 검사 통과 횟수를 누적합니다.
        passed += 1

        # 3) 공격 한 번으로 HP 감소 및 턴 증가
        # slime에 대해 정상 공격을 한 번 실행합니다.
        attack = await client.call_tool("attack_enemy", {"target": "slime"})
        # 공격 후 HP와 턴을 묶어 확인하는 비교식을 시작합니다.
        assert (attack.structured_content["state"]["enemy_hp"],
                # HP 40과 턴 1을 모두 만족하는지 검사합니다.
                attack.structured_content["state"]["turn"]) == (40, 1)
        # 세 번째 검사 통과 횟수를 누적합니다.
        passed += 1

        # 4) 잘못된 target에는 상태 변경이 없음
        # 허용하지 않는 대상 dragon에 대한 공격을 시험합니다.
        invalid = await client.call_tool("attack_enemy", {"target": "dragon"})
        # 잘못된 대상 오류 코드가 반환되는지 검사합니다.
        assert invalid.structured_content["error_code"] == "INVALID_TARGET"
        # 거부 후 서버 상태가 그대로인지 다시 조회합니다.
        state = await client.call_tool("get_game_status", {})
        # 오류 요청이 몬스터 HP를 변경하지 않았는지 검사합니다.
        assert state.structured_content["state"]["enemy_hp"] == 40
        # 네 번째 검사 통과 횟수를 누적합니다.
        passed += 1

        # 5) HP가 100이면 포션을 낭비하지 않음
        # 최대 HP에서 포션을 사용하려 시도합니다.
        heal = await client.call_tool("heal_player", {})
        # HP_ALREADY_FULL 오류를 반환하는지 확인합니다.
        assert heal.structured_content["error_code"] == "HP_ALREADY_FULL"
        # 서버의 포션 수를 다시 조회합니다.
        state = await client.call_tool("get_game_status", {})
        # 사용할 필요 없는 포션이 그대로 2개인지 확인합니다.
        assert state.structured_content["state"]["potions"] == 2
        # 다섯 번째 검사 통과 횟수를 누적합니다.
        passed += 1

        # 6) 총 3번 공격하면 승리
        # 남은 슬라임 HP를 줄이기 위해 두 번째 공격을 실행합니다.
        await client.call_tool("attack_enemy", {"target": "slime"})
        # 세 번째 공격을 실행해 승리 상태를 만들고 결과를 받습니다.
        win = await client.call_tool("attack_enemy", {"target": "slime"})
        # 몬스터 HP가 0인지 확인합니다.
        assert win.structured_content["state"]["enemy_hp"] == 0
        # 전투 상태가 win으로 설정되었는지 확인합니다.
        assert win.structured_content["state"]["outcome"] == "win"
        # 여섯 번째 검사 통과 횟수를 누적합니다.
        passed += 1

        # 7) 게임 종료 후에는 추가 공격이 허용되지 않음
        # 승리한 뒤에도 공격이 되는지 시험합니다.
        extra = await client.call_tool("attack_enemy", {"target": "slime"})
        # 이미 끝난 게임이므로 GAME_OVER가 반환되는지 검사합니다.
        assert extra.structured_content["error_code"] == "GAME_OVER"
        # 일곱 번째 검사 통과 횟수를 누적합니다.
        passed += 1

    # 통과한 전체 검사 수를 출력합니다.
    print(f"✅ 총 {passed}개 체크 통과")

# 비동기 테스트 함수를 실행합니다.
await run_contract_tests()


✅ 총 7개 체크 통과


## 9. 학습 정리: MCP가 맡는 역할

| 구분 | 이 실습에서 확인한 역할 |
|---|---|
| **Tool** | 함수 목록을 공개하고 `tools/call`로 실제 실행 |
| **Resource** | 고정 URI의 몬스터 도감 데이터를 조회 |
| **Prompt** | 전투 전략 지시문 템플릿을 반환 (실행하지 않음) |
| **Client** | 서버의 도구를 발견하고 요청·결과를 주고받음 |
| **Server** | 도구를 제공하며 Python 함수로 게임 상태를 변경함 |
| **Host** | 실제 앱에서는 사용자 상호작용, 서버 신뢰 및 승인을 담당함. 이 실습은 Host UI를 구현하지 않음 |

**핵심:** 실습 1의 Python 함수를 MCP 표준 인터페이스로 제공했습니다. **MCP는 연결 방식의 표준이며, 모델의 판단·권한 검사를 대신하지 않습니다.** 다음 실습 3에서는 이 도구들을 LangGraph의 노드에서 호출하도록 확장합니다.

### 참고 문서
- MCP Python SDK Client: https://py.sdk.modelcontextprotocol.io/client/
- MCP Python SDK Server: https://py.sdk.modelcontextprotocol.io/server/
- 강의자료: 9장 Tool Calling · MCP · LangGraph (MCP 서버 구현 및 In-memory Client)

**Colab 실행이 실패한다면:** 설치된 `mcp` 버전 확인 → 위에서부터 재실행 → 오류가 난 셀과 오류 메시지를 함께 확인하세요.